In [2]:
import os

files = ["yellow_tripdata_2025-11.parquet", "taxi_zone_lookup.csv"]
urls = [
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet",
    "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
]

for file, url in zip(files, urls):
    if not os.path.exists(file):
        print(f"Downloading {file}...")
        os.system(f"wget -q {url}")
    else:
        print(f"{file} already exists, skipping download.")

yellow_tripdata_2025-11.parquet already exists, skipping download.
taxi_zone_lookup.csv already exists, skipping download.


### Question 1: Install Spark and PySpark

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[2]") \
    .appName("Homework") \
    .getOrCreate()

df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

print(f"Spark version: {spark.version}")
print(f"Rows: {df.count():,}")


Spark version: 4.0.1


Rows: 4,181,444


### Question 2: Yellow November 2025

In [10]:
import os
import glob

# Repartition and save to parquet
df.repartition(4).write.mode("overwrite").parquet("yellow_tripdata_2025-11-partitioned")

# Check actual file sizes on disk
parquet_files = glob.glob("yellow_tripdata_2025-11-partitioned/*.parquet")
sizes_mb = [os.path.getsize(f) / (1024 * 1024) for f in parquet_files]

print(f"Parquet files found: {len(parquet_files)}")
for f, s in zip(parquet_files, sizes_mb):
    print(f"  {os.path.basename(f)}: {s:.2f} MB")
print(f"\nMean partition size: {sum(sizes_mb) / len(sizes_mb):.2f} MB")


Parquet files found: 4
  part-00001-294de0ae-97e0-48d9-8fa8-50424e751465-c000.snappy.parquet: 25.34 MB
  part-00000-294de0ae-97e0-48d9-8fa8-50424e751465-c000.snappy.parquet: 25.35 MB
  part-00002-294de0ae-97e0-48d9-8fa8-50424e751465-c000.snappy.parquet: 25.33 MB
  part-00003-294de0ae-97e0-48d9-8fa8-50424e751465-c000.snappy.parquet: 25.34 MB

Mean partition size: 25.34 MB


### Question 3: Count records

In [11]:
# How many taxi trips were there on the 15th of November?

from pyspark.sql.functions import col, to_date

df = spark.read.parquet("yellow_tripdata_2025-11.parquet")
df = df.withColumn("pickup_date", to_date(col("tpep_pickup_datetime")))

trips_on_15th = df.filter(col("pickup_date") == "2025-11-15").count()

print(f"Number of trips on 2025-11-15: {trips_on_15th:,}")

Number of trips on 2025-11-15: 162,604


### Question 4: Longest trip

In [20]:
# What is the length of the longest trip in the dataset in hours?

from pyspark.sql.functions import unix_timestamp, max as spark_max

df = spark.read.parquet("yellow_tripdata_2025-11.parquet")
df = df.withColumn(
    "trip_duration_hours",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 3600
)

longest_trip = df.select(spark_max("trip_duration_hours")).collect()[0][0]
print(f"Longest trip: {longest_trip:.1f} hours")


Longest trip: 90.6 hours


### Question 5: User Interface

In [ ]:
locahost:4040

### Question 6: Least frequent pickup location zone

In [ ]:
from pyspark.sql.functions import count, asc

df = spark.read.parquet("yellow_tripdata_2025-11.parquet")
df_zone = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)

# Count trips per pickup location
df_pickup_counts = df.groupBy("PULocationID").agg(count("*").alias("trip_count"))

# Join with zone lookup to get zone names
df_result = df_pickup_counts.join(
    df_zone, df_pickup_counts["PULocationID"] == df_zone["LocationID"]
)

# Find the least frequent pickup zone
least_frequent = df_result.orderBy(asc("trip_count")).select("Zone", "trip_count").limit(1)
least_frequent.show(truncate=False)


AssertionError: all exprs should be Column